# 7.16 舊例子混回去有用嗎？


只練新任務B時，舊任務A參數可能被改得不再合用。一個直接辦法是在新訓練中保留一些舊題，讓模型仍收到A的更新意見；這叫replay（回放）。它不是播放舊的梯度檔，而是再把舊資料放進當前訓練，重新用新參數計算預測、代價與梯度。

[7.15](https://birdhackor.github.io/tiny-perceptron-vlm/7.15.html) 已定義A/B前後評估，[5.16](https://birdhackor.github.io/tiny-perceptron-vlm/5.16.html) 又說明筆數比例與有效token比例不同。比較B-only與replay時，應該固定總更新或總有效答案預算，否則A+B組如果多訓練一倍，改善可能來自額外計算。還要報告兩任務各自表現，避免保住A但B幾乎不學也被稱總成功。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
a = [
    {"source": "A", "text": "舊題一", "answer_tokens": 2},
    {"source": "A", "text": "舊題二", "answer_tokens": 2},
]
b = [
    {"source": "B", "text": "新題一", "answer_tokens": 2},
    {"source": "B", "text": "新題二", "answer_tokens": 2},
]
recipe = a + b
print("每批來源", [record["source"] for record in recipe])
print("有效答案預算", sum(record["answer_tokens"] for record in recipe))
print("待測B-only與replay各自的A/B成績")

輸入兩筆A與兩筆B，每筆假設兩個有效答案token，合成四筆batch，所以來源A、A、B、B，token總數8，兩邊各半。text是來源示意名，不是完整訓練對話；真正數據應使用實際問答與測得答案長度。這段只計配方與預算，未訓練，不輸出虛構保留能力曲線。

固定batch四筆時，B-only可用四筆B共八token，replay用兩筆A兩筆B同樣八token；總預算一致，但replay新任務資料份額少一些。這是方法真實的取捨，不能一邊讓B兩筆再加A兩筆額外跑一輪、另一邊只給B原兩筆，然後說都是同樣成本。

舊資料也要有足夠覆蓋，不應把同一例複製很多次就當廣泛提醒。高品質、代表舊任務關鍵規則的樣本，可能比重複空泛句有用；選擇需用留出A題驗證，而非只看回放訓練題loss。B同樣要測新組合，防止為了保A只記住少量新題。

回放不是保證任何遺忘都被解決。A、B目標可能衝突，參數容量有限，比例或步幅也可能不合適；應保留不同配方、種子與預算的實際結果。它提供一份重新平衡共享參數的訓練信號，效果取決於資料與整個流程。

練習把recipe改成 `[a[0],b[0],b[1],b[0]]`，先預測來源25:75、有效token仍8，再執行核對。這份小練習為保持四筆而重複一B例，實際實驗應換不同B資料；它不增加資料覆蓋。若把B答案長度改4，重算token比例後就不再是25:75，這能把回放配方接回本章的有效監督統計。
